### Step 1 — Load Raw Data

Load all four input files into separate raw dataframes for inspection.

In [25]:
import pandas as pd

# ── Load Raw Data ──────────────────────────────────────────────────────────────
df_raw_consumption = pd.read_excel(r"C:\Users\Amey\Desktop\Amey\Python\Consumption_data.xlsx")
df_raw_equip_no    = pd.read_excel(r"C:\Users\Amey\Desktop\Amey\Python\Equipment No and OBR Data.xlsx")
df_raw_temp        = pd.read_csv(r"C:\Users\Amey\Desktop\Amey\Python\clean_temperature_data.csv")
df_raw_region      = pd.read_csv(r"C:\Users\Amey\Desktop\Amey\Python\cities.csv")

# ── Quick Verification ─────────────────────────────────────────────────────────
for name, df in [("df_raw_consumption", df_raw_consumption),
                 ("df_raw_equip_no",    df_raw_equip_no),
                 ("df_raw_temp",        df_raw_temp),
                 ("df_raw_region",      df_raw_region)]:
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Columns: {df.columns.tolist()}\n")

df_raw_consumption: 64,895 rows × 18 columns
  Columns: ['Material', 'Material Description', 'MvT', 'SLoc', 'User name', 'Sales Ord.', '   Quantity', 'EUn', 'Pstng Date', 'Reference', 'PO', 'WBS Elem.', 'Entry Date', 'Doc. Date', 'Order', 'Vendor', 'Plnt', 'Text']

df_raw_equip_no: 20,283 rows × 18 columns
  Columns: ['CN', 'NC #', 'Order', 'Creation date', 'Status', 'Requested deliv.date', 'Handover to customer', 'Short Text', 'Equipment', 'Equi. long address', '    Net value', 'Sell hours', 'Technician name', 'Technician 2 No', 'NC Aging', 'Risk Flag', 'NC Rej. by', 'NC Rej. on']

df_raw_temp: 202,797 rows × 11 columns
  Columns: ['SLOC', 'Location', 'Date', 'Tavg', 'Tmax', 'Tmin', 'RH', 'Month', 'Year', 'Season', 'Delta_T']

df_raw_region: 87 rows × 4 columns
  Columns: ['SLOC', 'City', 'Location', 'Region']



### Step 2 — Remove Unnecessary Columns

Drop all columns not required for analysis from the Consumption, Equipment/OBR, and Region dataframes.  
The temperature dataframe already contains only the required columns so it is saved as-is.

In [26]:
# ── df_raw_consumption: Drop unnecessary columns ───────────────────────────────
cols_to_drop_consumption = [
    'Material Description', 'MvT', 'User name', 'EUn', 'Reference',
    'PO', 'WBS Elem.', 'Entry Date', 'Doc. Date', 'Order', 'Vendor', 'Plnt', 'Text'
]
df_req_col_consumption = df_raw_consumption.drop(columns=cols_to_drop_consumption)

# ── df_raw_equip_no: Drop unnecessary columns ──────────────────────────────────
cols_to_drop_equip_no = [
    'CN', 'NC #', 'Creation date', 'Status', 'Requested deliv.date',
    'Handover to customer', 'Short Text', 'Equi. long address', '    Net value',
    'Sell hours', 'Technician 2 No', 'NC Aging', 'Risk Flag', 'NC Rej. by', 'NC Rej. on'
]
df_req_col_equip_no = df_raw_equip_no.drop(columns=cols_to_drop_equip_no)

# ── df_raw_temp: No columns to drop, save as-is ───────────────────────────────
df_req_col_temp = df_raw_temp.copy()

# ── df_raw_region: Drop City and Location columns ─────────────────────────────
df_req_region = df_raw_region.drop(columns=['City', 'Location'])

# ── Verification ───────────────────────────────────────────────────────────────
for name, df in [("df_req_col_consumption", df_req_col_consumption),
                 ("df_req_col_equip_no",    df_req_col_equip_no),
                 ("df_req_col_temp",        df_req_col_temp),
                 ("df_req_region",          df_req_region)]:
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Columns: {df.columns.tolist()}\n")

df_req_col_consumption: 64,895 rows × 5 columns
  Columns: ['Material', 'SLoc', 'Sales Ord.', '   Quantity', 'Pstng Date']

df_req_col_equip_no: 20,283 rows × 3 columns
  Columns: ['Order', 'Equipment', 'Technician name']

df_req_col_temp: 202,797 rows × 11 columns
  Columns: ['SLOC', 'Location', 'Date', 'Tavg', 'Tmax', 'Tmin', 'RH', 'Month', 'Year', 'Season', 'Delta_T']

df_req_region: 87 rows × 2 columns
  Columns: ['SLOC', 'Region']



### Step 3 — Standardise Column Names and Data Types

Strip whitespace from all column names first, then rename columns to match the agreed schema  
and enforce the correct data types across all four dataframes.  
Note: Equipment Number is kept as String because some values contain hyphens (e.g. 10687716-1).

In [27]:
# ── Strip whitespace from ALL column names first (handles hidden spaces) ───────
df_req_col_consumption.columns = df_req_col_consumption.columns.str.strip()
df_req_col_equip_no.columns    = df_req_col_equip_no.columns.str.strip()
df_req_col_temp.columns        = df_req_col_temp.columns.str.strip()
df_req_region.columns          = df_req_region.columns.str.strip()

# ── df_req_col_consumption ─────────────────────────────────────────────────────
df_req_col_consumption = df_req_col_consumption.rename(columns={
    'Sales Ord.' : 'Order',
    'Pstng Date' : 'Date'
})

df_req_col_consumption['Material'] = df_req_col_consumption['Material'].astype(str)
df_req_col_consumption['SLoc']     = df_req_col_consumption['SLoc'].astype(str)
df_req_col_consumption['Order']    = df_req_col_consumption['Order'].astype(str)
df_req_col_consumption['Quantity'] = df_req_col_consumption['Quantity'].astype(int)
df_req_col_consumption['Date']     = pd.to_datetime(df_req_col_consumption['Date'])

# ── df_req_col_equip_no ────────────────────────────────────────────────────────
df_req_col_equip_no = df_req_col_equip_no.rename(columns={
    'Equipment'      : 'Equipment Number',
    'Technician name': 'Technician Name'
})

df_req_col_equip_no['Order']            = df_req_col_equip_no['Order'].astype(str)
df_req_col_equip_no['Equipment Number'] = df_req_col_equip_no['Equipment Number'].astype(str)
df_req_col_equip_no['Technician Name']  = df_req_col_equip_no['Technician Name'].astype(str)

# ── df_req_col_temp ────────────────────────────────────────────────────────────
df_req_col_temp = df_req_col_temp.rename(columns={
    'SLOC' : 'SLoc'
})

df_req_col_temp['SLoc']     = df_req_col_temp['SLoc'].astype(str)
df_req_col_temp['Location'] = df_req_col_temp['Location'].astype(str)
df_req_col_temp['Date']     = pd.to_datetime(df_req_col_temp['Date'])
df_req_col_temp['Tavg']     = df_req_col_temp['Tavg'].astype(float)
df_req_col_temp['Tmax']     = df_req_col_temp['Tmax'].astype(float)
df_req_col_temp['Tmin']     = df_req_col_temp['Tmin'].astype(float)
df_req_col_temp['RH']       = df_req_col_temp['RH'].astype(float)
df_req_col_temp['Month']    = df_req_col_temp['Month'].astype(int)
df_req_col_temp['Year']     = df_req_col_temp['Year'].astype(int)
df_req_col_temp['Season']   = df_req_col_temp['Season'].astype(str)
df_req_col_temp['Delta_T']  = df_req_col_temp['Delta_T'].astype(float)

# ── df_req_region ──────────────────────────────────────────────────────────────
df_req_region = df_req_region.rename(columns={
    'SLOC' : 'SLoc'
})

df_req_region['SLoc']   = df_req_region['SLoc'].astype(str)
df_req_region['Region'] = df_req_region['Region'].astype(str)

# ── Verification ───────────────────────────────────────────────────────────────
for name, df in [("df_req_col_consumption", df_req_col_consumption),
                 ("df_req_col_equip_no",    df_req_col_equip_no),
                 ("df_req_col_temp",        df_req_col_temp),
                 ("df_req_region",          df_req_region)]:
    print(f"{'='*40}")
    print(f"{name}")
    print(f"{'='*40}")
    print(df.dtypes)
    print()
    display(df.head(2))
    print()

df_req_col_consumption
Material               str
SLoc                   str
Order                  str
Quantity             int64
Date        datetime64[us]
dtype: object



,Material,SLoc,Order,Quantity,Date
0,111,5043,68160889,-1,2026-01-01
1,111,5043,68160889,1,2026-01-01



df_req_col_equip_no
Order               str
Equipment Number    str
Technician Name     str
dtype: object



,Order,Equipment Number,Technician Name
0,68631760,10853186,Sanjay Name
1,68526738,10667512,Saurabh Borle



df_req_col_temp
SLoc                   str
Location               str
Date        datetime64[us]
Tavg               float64
Tmax               float64
Tmin               float64
RH                 float64
Month                int64
Year                 int64
Season                 str
Delta_T            float64
dtype: object



,SLoc,Location,Date,Tavg,Tmax,Tmin,RH,Month,Year,Season,Delta_T
0,5015,Dadra and Nagar Ha,2020-01-01,18.76,26.88,12.06,69.16,1,2020,Winter,14.82
1,5015,Dadra and Nagar Ha,2020-01-02,19.76,27.71,14.27,75.41,1,2020,Winter,13.44



df_req_region
SLoc      str
Region    str
dtype: object



,SLoc,Region
0,5015,East
1,5019,West 1


### Step 4 — Create df_combined

Merge all dataframes into a single combined dataframe using left joins.  
- Equipment Number and Technician Name are mapped from df_req_col_equip_no using Order as the key.  
- Weather columns are mapped from df_req_col_temp using SLoc + Date as the key.  
- Region is mapped from df_req_region using SLoc as the key.  
All joins are left joins so every consumption row is retained even without a match.

In [28]:
# ── Merge 1: Consumption ← Equipment/OBR on Order ─────────────────────────────
df_combined = df_req_col_consumption.merge(
    df_req_col_equip_no,
    on  = 'Order',
    how = 'left'
)

# ── Merge 2: Result ← Temperature on SLoc + Date ──────────────────────────────
df_combined = df_combined.merge(
    df_req_col_temp,
    on  = ['SLoc', 'Date'],
    how = 'left'
)

# ── Merge 3: Result ← Region on SLoc ──────────────────────────────────────────
df_combined = df_combined.merge(
    df_req_region,
    on  = 'SLoc',
    how = 'left'
)

# ── Reorder columns as per agreed structure ────────────────────────────────────
df_combined = df_combined[[
    'Material', 'SLoc', 'Order', 'Quantity',
    'Equipment Number', 'Technician Name',
    'Tavg', 'Tmax', 'Tmin', 'RH', 'Delta_T',
    'Date', 'Month', 'Year', 'Season', 'Location', 'Region'
]]

# ── Verification ───────────────────────────────────────────────────────────────
print(f"Shape   : {df_combined.shape[0]:,} rows × {df_combined.shape[1]} columns")
print(f"Columns : {df_combined.columns.tolist()}")
print()
display(df_combined.head(5))

Shape   : 64,895 rows × 17 columns
Columns : ['Material', 'SLoc', 'Order', 'Quantity', 'Equipment Number', 'Technician Name', 'Tavg', 'Tmax', 'Tmin', 'RH', 'Delta_T', 'Date', 'Month', 'Year', 'Season', 'Location', 'Region']



,Material,SLoc,Order,Quantity,Equipment Number,Technician Name,Tavg,Tmax,Tmin,RH,Delta_T,Date,Month,Year,Season,Location,Region
0,111,5043,68160889,-1,NaN,NaN,24.68,27.36,22.93,85.81,4.43,2026-01-01,1,2026,Winter,Chennai,South
1,111,5043,68160889,1,NaN,NaN,24.68,27.36,22.93,85.81,4.43,2026-01-01,1,2026,Winter,Chennai,South
2,100,5030,68165456,-1,11565095,Kamal Kant,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1
3,111,5030,68165435,-1,NaN,NaN,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1
4,111,5030,68165114,-1,NaN,NaN,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1


### Step 5 — Data Quality Checks

Check for missing and duplicate Order numbers in df_combined, and missing SLoc values  
that could not be matched to the temperature/region data.  
Results are saved to separate CSV files for review and correction.

In [ ]:
# ── Step 5: Data Quality Checks ───────────────────────────────────────────────

import os
os.makedirs("Logs", exist_ok=True)

# ── Missing Orders ─────────────────────────────────────────────────────────────
df_missing_orders = df_combined[
    df_combined['Order'].isna() | (df_combined['Order'] == 'nan')
]
df_missing_orders.to_csv("Logs/missing_order.csv", index=False)
print(f"Missing Orders       : {len(df_missing_orders):,} rows → Logs/missing_order.csv")

# ── Missing SLoc ───────────────────────────────────────────────────────────────
df_missing_sloc = df_combined[
    df_combined['Tavg'].isna() | df_combined['Region'].isna()
]
df_missing_sloc.to_csv("Logs/missing_sloc.csv", index=False)
print(f"Missing SLoc         : {len(df_missing_sloc):,} rows → Logs/missing_sloc.csv")

# ── Merge Fan-out Check ────────────────────────────────────────────────────────
equip_per_order = df_req_col_equip_no.groupby('Order')['Equipment Number'].nunique()
fanout_orders   = equip_per_order[equip_per_order > 1]
if len(fanout_orders) > 0:
    fanout_orders.reset_index().to_csv("Logs/fanout_orders.csv", index=False)
    print(f"Merge Fan-out Orders : {len(fanout_orders):,} orders → Logs/fanout_orders.csv ⚠️")
else:
    print(f"Merge Fan-out Orders : 0 (clean)")

# ── Multi-line Orders (expected SAP behaviour) ─────────────────────────────────
order_counts      = df_combined.groupby('Order').size()
multi_line_orders = order_counts[order_counts > 1]
print(f"Multi-line Orders    : {len(multi_line_orders):,} orders with >1 material line (expected)")

# ── True Duplicates: identical rows entered twice in SAP ──────────────────────
# Negatives are valid SAP reversals — kept as-is. Only exact duplicate rows removed.
rows_before = len(df_combined)

df_true_duplicates = df_combined[
    df_combined.duplicated(
        subset=['Order', 'Material', 'SLoc', 'Date', 'Quantity'],
        keep=False
    )
].sort_values(['Order', 'Material'])

df_true_duplicates.to_csv("Logs/duplicate_rows.csv", index=False)
print(f"True Duplicate Rows  : {len(df_true_duplicates):,} rows → Logs/duplicate_rows.csv")

# ── Remove duplicates, keep first occurrence ───────────────────────────────────
df_combined = df_combined.drop_duplicates(
    subset=['Order', 'Material', 'SLoc', 'Date', 'Quantity'],
    keep='first'
).reset_index(drop=True)

# ── Final Summary ──────────────────────────────────────────────────────────────
print(f"\n{'='*45}")
print(f"Total rows (raw)     : {rows_before:,}")
print(f"Missing Orders       : {len(df_missing_orders):,}")
print(f"Missing SLoc/Temp    : {len(df_missing_sloc):,}")
print(f"Duplicates removed   : {rows_before - len(df_combined):,}")
print(f"df_combined (clean)  : {len(df_combined):,} rows")
print(f"{'='*45}")

display(df_combined.head(5))

Missing Orders       : 60 rows → Logs/missing_order.csv
Missing SLoc         : 0 rows → Logs/missing_sloc.csv
Merge Fan-out Orders : 0 (clean)
Multi-line Orders    : 6,001 orders with >1 material line (expected)
True Duplicate Rows  : 0 rows → Logs/duplicate_rows.csv

Total rows (raw)     : 64,807
Missing Orders       : 60
Missing SLoc/Temp    : 0
Duplicates removed   : 0
df_combined (clean)  : 64,807 rows


,Material,SLoc,Order,Quantity,Equipment Number,Technician Name,Tavg,Tmax,Tmin,RH,Delta_T,Date,Month,Year,Season,Location,Region
0,111,5043,68160889,-1,NaN,NaN,24.68,27.36,22.93,85.81,4.43,2026-01-01,1,2026,Winter,Chennai,South
1,111,5043,68160889,1,NaN,NaN,24.68,27.36,22.93,85.81,4.43,2026-01-01,1,2026,Winter,Chennai,South
2,100,5030,68165456,-1,11565095,Kamal Kant,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1
3,111,5030,68165435,-1,NaN,NaN,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1
4,111,5030,68165114,-1,NaN,NaN,11.47,18.40,5.11,70.89,13.29,2026-01-03,1,2026,Winter,Delhi,North1


In [32]:
# ── Step 6: Export to Excel — Three Representations ───────────────────────────

# ── Sheet 1: Combined Master Data (transaction level) ─────────────────────────
df_sheet1 = df_combined.copy()

# ── Sheet 2: Monthly Consumption Pivot (Material × Month-Year) ────────────────
# Rows = Material, Columns = Jan-20, Feb-20, ... (chronological)
df_pivot = df_combined.copy()
df_pivot['Month_Year'] = pd.to_datetime(df_pivot['Date']).dt.to_period('M')

df_sheet2 = (
    df_pivot
    .groupby(['Material', 'Month_Year'])['Quantity']
    .sum()
    .unstack('Month_Year')
    .fillna(0)
    .astype(int)
)
df_sheet2.columns = [str(c) for c in df_sheet2.columns]  # e.g. '2020-01'
df_sheet2 = df_sheet2.reset_index()

# ── Sheet 3: Material-wise Monthly Aggregation with Environmental Columns ──────
# Group by Material + Month + Year, sum Quantity, average the weather columns
df_sheet3 = (
    df_combined
    .groupby(['Material', 'Year', 'Month', 'Season', 'Region'], as_index=False, dropna=False)
    .agg(
        Quantity  = ('Quantity', 'sum'),
        Tavg      = ('Tavg',     'mean'),
        Tmax      = ('Tmax',     'mean'),
        Tmin      = ('Tmin',     'mean'),
        RH        = ('RH',       'mean'),
        Delta_T   = ('Delta_T',  'mean'),
    )
    .round(2)
    .sort_values(['Material', 'Year', 'Month'])
    .reset_index(drop=True)
)

# ── Write all three sheets to one Excel file ───────────────────────────────────
output_path = "Combined_Master_Data.xlsx"

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_sheet1.to_excel(writer, sheet_name='Master Data',         index=False)
    df_sheet2.to_excel(writer, sheet_name='Monthly Pivot',       index=False)
    df_sheet3.to_excel(writer, sheet_name='Material Monthly Agg', index=False)

# ── Apply formatting ───────────────────────────────────────────────────────────
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HEADER_FILL  = PatternFill("solid", fgColor="1F4E79")   # dark blue
HEADER_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
CELL_FONT    = Font(name="Arial", size=10)
ALT_FILL     = PatternFill("solid", fgColor="DCE6F1")   # light blue
BORDER_SIDE  = Side(style='thin', color='B8CCE4')
CELL_BORDER  = Border(bottom=BORDER_SIDE)
CENTER_ALIGN = Alignment(horizontal='center', vertical='center')

def format_sheet(ws):
    # Header row
    for cell in ws[1]:
        cell.font      = HEADER_FONT
        cell.fill      = HEADER_FILL
        cell.alignment = CENTER_ALIGN

    # Data rows — alternating fill + font
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        fill = ALT_FILL if row_idx % 2 == 0 else PatternFill()
        for cell in row:
            cell.font      = CELL_FONT
            cell.fill      = fill
            cell.border    = CELL_BORDER
            cell.alignment = Alignment(horizontal='center', vertical='center')

    # Auto-fit column widths
    for col in ws.columns:
        max_len = max((len(str(c.value)) if c.value is not None else 0) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 30)

    # Freeze header row
    ws.freeze_panes = "A2"

wb = load_workbook(output_path)
for sheet_name in wb.sheetnames:
    format_sheet(wb[sheet_name])
wb.save(output_path)

print(f"Exported: {output_path}")
print(f"  Sheet 1 — Master Data          : {len(df_sheet1):,} rows")
print(f"  Sheet 2 — Monthly Pivot        : {len(df_sheet2):,} materials × {len(df_sheet2.columns)-1} months")
print(f"  Sheet 3 — Material Monthly Agg : {len(df_sheet3):,} rows")

Exported: Combined_Master_Data.xlsx
  Sheet 1 — Master Data          : 64,807 rows
  Sheet 2 — Monthly Pivot        : 12 materials × 76 months
  Sheet 3 — Material Monthly Agg : 2,816 rows
